In [2]:
import pandas as pd

In [ ]:
data = pd.read_csv("proteinbase_all_data_28_01_2026.csv")
data

In [ ]:
data

In [ ]:
import json

def unpack_evaluations(df):
    """
    Unpacks all metric fields from the evaluations JSON column into separate columns.
    
    Creates:
    - Suffix-less columns for metrics without a target (e.g. 'esmfold_plddt').
    - Target-specific columns '{metric}_{target}' (e.g. 'kd_human-serum-albumin').
    - Simple '{metric}' columns mapped to the primary/intended designed target.
    """
    # 1. Helper function to extract primary designed target
    def extract_primary_target(evals_json):
        if pd.isna(evals_json) or not isinstance(evals_json, str):
            return None
        try:
            evals = json.loads(evals_json)
            targets = []
            for item in evals:
                if isinstance(item, dict) and "target" in item:
                    t = item["target"]
                    if t and t not in targets:
                        targets.append(t)
            if not targets:
                return None
            control_proteins = {"human-serum-albumin", "fcrn", "hsa"}
            if len(targets) > 1:
                filtered = [t for t in targets if t.lower() not in control_proteins]
                if filtered:
                    targets = filtered
            return targets[0] if len(targets) == 1 else ";".join(targets)
        except:
            return None

    print("Identifying primary targets...")
    df["primary_target"] = df["evaluations"].apply(extract_primary_target)

    print("Unpacking JSON metrics into columns...")
    unpacked_rows = []
    for _, row in df.iterrows():
        row_dict = {}
        evals_json = row["evaluations"]
        prim_target = row["primary_target"]
        
        if isinstance(evals_json, str) and not pd.isna(evals_json):
            try:
                evals = json.loads(evals_json)
                for item in evals:
                    if isinstance(item, dict) and "metric" in item:
                        metric = item["metric"]
                        val = item.get("value")
                        target = item.get("target")
                        
                        # 1. Target-specific column (e.g. kd_nipah-glycoprotein-g)
                        if target:
                            row_dict[f"{metric}_{target}"] = val
                            # 2. General primary metric column (e.g. kd)
                            if target == prim_target:
                                row_dict[metric] = val
                        else:
                            # 3. Suffix-less column for metrics without a target
                            row_dict[metric] = val
            except:
                pass
        unpacked_rows.append(row_dict)
        
    unpacked_df = pd.DataFrame(unpacked_rows, index=df.index)
    res_df = pd.concat([df.drop(columns=["evaluations"]), unpacked_df], axis=1)
    print(f"Success! Expanded dataframe shape to: {res_df.shape}")
    return res_df

# Run the unpacking function
df_unpacked = unpack_evaluations(data)
df_unpacked.head()

In [ ]:
# Print list of some newly created columns
print("=== Unpacked Metric Columns (Subset) ===")
unpacked_cols = [c for c in df_unpacked.columns if c not in data.columns]
print(f"Total new columns created: {len(unpacked_cols)}")
print("\nSample metrics (without targets):")
print([c for c in unpacked_cols if "_" not in c][:15])
print("\nSample target-specific metrics:")
print([c for c in unpacked_cols if "_" in c][:15])

In [3]:
cleaned = pd.read_csv("cleaned_final.csv")
cleaned

,id,name,sequence,author,designMethod,type,metric,target,value_type,unit,value
0,azure-wolf-maple,control-6CMI_3,EVQLVQSGAEVKKRGSSVKVSCKSSGGTFSNYAINWVRQAPGQGLE...,adaptyv-bio,NaN,experimental,binding_strength,nipah-glycoprotein-g,label,NaN,Strong
1,calm-panda-fern,control-7TXZ_2,EVKLEESGGGLVQPGGSMKLSCVASGFSFSYYWMNWVRQSPEKGLE...,adaptyv-bio,NaN,experimental,binding_strength,nipah-glycoprotein-g,label,NaN,Strong
2,deep-heron-rose,control-ephrin-B2,KSIVLEPIYWNSSNSKFLPGQGLVLYPQIGDKLDIICPKVDSKTVG...,adaptyv-bio,NaN,experimental,binding_strength,nipah-glycoprotein-g,label,NaN,Strong
3,ivory-orca-fern,control-8K3C_2,EVQLVQSGGGLVQPGGSLRLSCAASGFTVSSNYMSWVRQAPGKGLE...,adaptyv-bio,NaN,experimental,binding_strength,nipah-glycoprotein-g,label,NaN,Strong
4,rough-bison-maple,QVQLQESGPGVVKPSETLSLTCAVSGGSISDTYRWSWIRQPPGKGL...,QVQLQESGPGVVKPSETLSLTCAVSGGSISDTYRWSWIRQPPGKGL...,andrew_huang,esmfold-rfdiffusion-and-rational-design-ikhggD...,experimental,binding_strength,nipah-glycoprotein-g,label,NaN,Strong
...,...,...,...,...,...,...,...,...,...,...,...
523,crimson-mole-quartz,SAADEAARAASLTLLDYLVERYTDPATSEEDKRFLLRLAERLVETY...,SAADEAARAASLTLLDYLVERYTDPATSEEDKRFLLRLAERLVETY...,escalante-bio,mosaic,experimental,binding_strength,pd-l1,label,NaN,Medium
524,young-bear-cypress,MEPTDEEKRGKYVAKIVAEEALKAYTETKDEEELNFLLIKLAALSE...,MEPTDEEKRGKYVAKIVAEEALKAYTETKDEEELNFLLIKLAALSE...,escalante-bio,mosaic,experimental,binding_strength,il7r,label,NaN,Medium
525,violet-moth-crystal,GPLSKAEYLEEQIKFVEENKDKKELVKPKLIETLKYVGYDEEAEYY...,GPLSKAEYLEEQIKFVEENKDKKELVKPKLIETLKYVGYDEEAEYY...,escalante-bio,mosaic,experimental,binding_strength,il7r,label,NaN,Strong
526,steady-bat-crystal,EVNCELLNKLAEPVAKELFPENDLAKISTFYAALFFAYEGALKGKP...,EVNCELLNKLAEPVAKELFPENDLAKISTFYAALFFAYEGALKGKP...,escalante-bio,mosaic,experimental,binding_strength,pd-l1,label,NaN,Weak


In [5]:
import os
import requests

# Dictionary mapping your specific target description slugs to their UniProt Accession IDs
uniprot_mapping = {
    'nipah-glycoprotein-g': 'Q9IH62',       # Nipah virus Attachment glycoprotein
    'fcrn': 'P55899',                       # Human IgG receptor FcRn large subunit 
    'human-pdgfr-beta': 'P09619',           # Human Platelet-derived growth factor receptor beta
    'human-mzb1-perp1': 'Q8WU39',           # Human Marginal zone B- and B1-cell-specific protein
    'human-idi2': 'Q9BXS1',                 # Human Isopentenyl-diphosphate delta-isomerase 2
    'il7r': 'P16871',                       # Human Interleukin-7 receptor subunit alpha
    'human-insulin-receptor': 'P06213',     # Human Insulin receptor
    'hnmt': 'P50135',                       # Human Histamine N-methyltransferase
    'pd-l1': 'Q9NZQ7',                      # Human Programmed cell death 1 ligand 1
    'human-pmvk': 'Q15126',                 # Human Phosphomevalonate kinase
    'human-phyh': 'O14832',                 # Human Phytanoyl-CoA hydroxylase
    'human-ambp': 'P02760',                 # Human AMBP protein
    'human-rfk': 'Q969G6',                  # Human Riboflavin kinase
    'egfr': 'P00533',                       # Human Epidermal growth factor receptor
    'mdm2': 'Q00987',                       # Human E3 ubiquitin-protein ligase Mdm2
    'fgf-r1': 'P11362',                     # Human Fibroblast growth factor receptor 1
    'spcas9': 'Q99ZW2',                     # Streptococcus pyogenes Cas9
    'ifnar2': 'P48551',                     # Human Interferon alpha/beta receptor subunit 2
    'der7': 'Q23841',                       # Dermatophagoides farinae Allergen Der f 7
    'der21': 'Q1A1R9'                       # Dermatophagoides pteronyssinus Allergen Der p 21
}

# Download target sequences if they do not exist locally
fasta_filepath = "actual_target_proteins.fasta"
target_sequences = {}

if not os.path.exists(fasta_filepath):
    print("Downloading target sequences from UniProt...")
    with open(fasta_filepath, 'w') as fasta_file:
        for target_name, accession in uniprot_mapping.items():
            url = f"https://rest.uniprot.org/uniprotkb/{accession}.fasta"
            try:
                response = requests.get(url)
                if response.status_code == 200:
                    lines = response.text.strip().split('\n')
                    header = f">{target_name} uniprot={accession}"
                    sequence = "".join(lines[1:])
                    target_sequences[target_name] = sequence
                    
                    fasta_file.write(header + '\n' + '\n'.join(lines[1:]) + '\n')
                    print(f"Successfully downloaded sequence for: {target_name}")
                else:
                    print(f"Failed to fetch sequence for: {target_name} (Status code: {response.status_code})")
            except Exception as e:
                print(f"Error fetching sequence for {target_name}: {e}")
else:
    print(f"Loading sequences from existing local file '{fasta_filepath}'...")
    current_target = None
    current_seq_lines = []
    with open(fasta_filepath, 'r') as fasta_file:
        for line in fasta_file:
            line = line.strip()
            if line.startswith('>'):
                if current_target:
                    target_sequences[current_target] = "".join(current_seq_lines)
                # Parse target_name from header
                header_parts = line[1:].split()
                current_target = header_parts[0]
                current_seq_lines = []
            elif line:
                current_seq_lines.append(line)
        if current_target:
            target_sequences[current_target] = "".join(current_seq_lines)

# Map primary_target column to the downloaded FASTA sequences
cleaned['target_sequence'] = cleaned['target'].map(target_sequences)
print(f"Successfully mapped target FASTA sequences to 'target_sequence' column in df_unpacked!")

# Show unique targets mapped and their sequences
cleaned[['target', 'target_sequence']].drop_duplicates().dropna().head(10)

Loading sequences from existing local file 'actual_target_proteins.fasta'...
Successfully mapped target FASTA sequences to 'target_sequence' column in df_unpacked!


,target,target_sequence
0,nipah-glycoprotein-g,MPAENKKVRFENTTSDKGKIPSKVIKSYYGTMDIKKINEGLLDSKI...
121,fcrn,MGVPRPQPWALGLLLFLLPGSLGAESHLSLLYHLTAVSSPAPGTPA...
130,human-pdgfr-beta,MRLPGAMPALALKGELLLLSLLLLLEPQISQGLVVTPPGPELVLNV...
131,human-mzb1-perp1,MRLSLPLLLLLLGAWAIPGGLGDRAPLTATAPQLDDEEMYSAHMPA...
136,human-idi2,MSDINLDWVDRRQLQRLEEMLIVVDENDKVIGADTKRNCHLNENIE...
143,il7r,MTILGTTFGMVFSLLQVVSGESGYAQNGDLEDAELDDYSFSCYSQL...
159,human-insulin-receptor,MATGGRRGAAAAPLLVAVAALLLGAAGHLYPGEVCPGMDIRNNLTR...
162,hnmt,MASSMRSLFSDHGKYVESFRRFLNHSTEHQCMQEFMDKKLPGIIGR...
189,pd-l1,MRIFAVFIFMTYWHLLNAFTVTVPKDLYVVEYGSNMTIECKFPVEK...
194,human-pmvk,MAPLGGAPRLVLLFSGKRKSGKDFVTEALQSRLGADVCAVLRLSGP...


In [6]:
cleaned

,id,name,sequence,author,designMethod,type,metric,target,value_type,unit,value,target_sequence
0,azure-wolf-maple,control-6CMI_3,EVQLVQSGAEVKKRGSSVKVSCKSSGGTFSNYAINWVRQAPGQGLE...,adaptyv-bio,NaN,experimental,binding_strength,nipah-glycoprotein-g,label,NaN,Strong,MPAENKKVRFENTTSDKGKIPSKVIKSYYGTMDIKKINEGLLDSKI...
1,calm-panda-fern,control-7TXZ_2,EVKLEESGGGLVQPGGSMKLSCVASGFSFSYYWMNWVRQSPEKGLE...,adaptyv-bio,NaN,experimental,binding_strength,nipah-glycoprotein-g,label,NaN,Strong,MPAENKKVRFENTTSDKGKIPSKVIKSYYGTMDIKKINEGLLDSKI...
2,deep-heron-rose,control-ephrin-B2,KSIVLEPIYWNSSNSKFLPGQGLVLYPQIGDKLDIICPKVDSKTVG...,adaptyv-bio,NaN,experimental,binding_strength,nipah-glycoprotein-g,label,NaN,Strong,MPAENKKVRFENTTSDKGKIPSKVIKSYYGTMDIKKINEGLLDSKI...
3,ivory-orca-fern,control-8K3C_2,EVQLVQSGGGLVQPGGSLRLSCAASGFTVSSNYMSWVRQAPGKGLE...,adaptyv-bio,NaN,experimental,binding_strength,nipah-glycoprotein-g,label,NaN,Strong,MPAENKKVRFENTTSDKGKIPSKVIKSYYGTMDIKKINEGLLDSKI...
4,rough-bison-maple,QVQLQESGPGVVKPSETLSLTCAVSGGSISDTYRWSWIRQPPGKGL...,QVQLQESGPGVVKPSETLSLTCAVSGGSISDTYRWSWIRQPPGKGL...,andrew_huang,esmfold-rfdiffusion-and-rational-design-ikhggD...,experimental,binding_strength,nipah-glycoprotein-g,label,NaN,Strong,MPAENKKVRFENTTSDKGKIPSKVIKSYYGTMDIKKINEGLLDSKI...
...,...,...,...,...,...,...,...,...,...,...,...,...
523,crimson-mole-quartz,SAADEAARAASLTLLDYLVERYTDPATSEEDKRFLLRLAERLVETY...,SAADEAARAASLTLLDYLVERYTDPATSEEDKRFLLRLAERLVETY...,escalante-bio,mosaic,experimental,binding_strength,pd-l1,label,NaN,Medium,MRIFAVFIFMTYWHLLNAFTVTVPKDLYVVEYGSNMTIECKFPVEK...
524,young-bear-cypress,MEPTDEEKRGKYVAKIVAEEALKAYTETKDEEELNFLLIKLAALSE...,MEPTDEEKRGKYVAKIVAEEALKAYTETKDEEELNFLLIKLAALSE...,escalante-bio,mosaic,experimental,binding_strength,il7r,label,NaN,Medium,MTILGTTFGMVFSLLQVVSGESGYAQNGDLEDAELDDYSFSCYSQL...
525,violet-moth-crystal,GPLSKAEYLEEQIKFVEENKDKKELVKPKLIETLKYVGYDEEAEYY...,GPLSKAEYLEEQIKFVEENKDKKELVKPKLIETLKYVGYDEEAEYY...,escalante-bio,mosaic,experimental,binding_strength,il7r,label,NaN,Strong,MTILGTTFGMVFSLLQVVSGESGYAQNGDLEDAELDDYSFSCYSQL...
526,steady-bat-crystal,EVNCELLNKLAEPVAKELFPENDLAKISTFYAALFFAYEGALKGKP...,EVNCELLNKLAEPVAKELFPENDLAKISTFYAALFFAYEGALKGKP...,escalante-bio,mosaic,experimental,binding_strength,pd-l1,label,NaN,Weak,MRIFAVFIFMTYWHLLNAFTVTVPKDLYVVEYGSNMTIECKFPVEK...


In [8]:
wanted_cols = ['sequence','target_sequence','value']
#get only the wanted
final_df = cleaned[wanted_cols]

In [9]:
final_df

,sequence,target_sequence,value
0,EVQLVQSGAEVKKRGSSVKVSCKSSGGTFSNYAINWVRQAPGQGLE...,MPAENKKVRFENTTSDKGKIPSKVIKSYYGTMDIKKINEGLLDSKI...,Strong
1,EVKLEESGGGLVQPGGSMKLSCVASGFSFSYYWMNWVRQSPEKGLE...,MPAENKKVRFENTTSDKGKIPSKVIKSYYGTMDIKKINEGLLDSKI...,Strong
2,KSIVLEPIYWNSSNSKFLPGQGLVLYPQIGDKLDIICPKVDSKTVG...,MPAENKKVRFENTTSDKGKIPSKVIKSYYGTMDIKKINEGLLDSKI...,Strong
3,EVQLVQSGGGLVQPGGSLRLSCAASGFTVSSNYMSWVRQAPGKGLE...,MPAENKKVRFENTTSDKGKIPSKVIKSYYGTMDIKKINEGLLDSKI...,Strong
4,QVQLQESGPGVVKPSETLSLTCAVSGGSISDTYRWSWIRQPPGKGL...,MPAENKKVRFENTTSDKGKIPSKVIKSYYGTMDIKKINEGLLDSKI...,Strong
...,...,...,...
523,SAADEAARAASLTLLDYLVERYTDPATSEEDKRFLLRLAERLVETY...,MRIFAVFIFMTYWHLLNAFTVTVPKDLYVVEYGSNMTIECKFPVEK...,Medium
524,MEPTDEEKRGKYVAKIVAEEALKAYTETKDEEELNFLLIKLAALSE...,MTILGTTFGMVFSLLQVVSGESGYAQNGDLEDAELDDYSFSCYSQL...,Medium
525,GPLSKAEYLEEQIKFVEENKDKKELVKPKLIETLKYVGYDEEAEYY...,MTILGTTFGMVFSLLQVVSGESGYAQNGDLEDAELDDYSFSCYSQL...,Strong
526,EVNCELLNKLAEPVAKELFPENDLAKISTFYAALFFAYEGALKGKP...,MRIFAVFIFMTYWHLLNAFTVTVPKDLYVVEYGSNMTIECKFPVEK...,Weak


In [10]:
final_df.to_csv("plain_training.csv",index=False)

In [3]:
final_df = pd.read_csv("plain_training.csv")

In [10]:
final_df

,sequence,target_sequence,value
0,EVQLVQSGAEVKKRGSSVKVSCKSSGGTFSNYAINWVRQAPGQGLE...,MPAENKKVRFENTTSDKGKIPSKVIKSYYGTMDIKKINEGLLDSKI...,Strong
1,EVKLEESGGGLVQPGGSMKLSCVASGFSFSYYWMNWVRQSPEKGLE...,MPAENKKVRFENTTSDKGKIPSKVIKSYYGTMDIKKINEGLLDSKI...,Strong
2,KSIVLEPIYWNSSNSKFLPGQGLVLYPQIGDKLDIICPKVDSKTVG...,MPAENKKVRFENTTSDKGKIPSKVIKSYYGTMDIKKINEGLLDSKI...,Strong
3,EVQLVQSGGGLVQPGGSLRLSCAASGFTVSSNYMSWVRQAPGKGLE...,MPAENKKVRFENTTSDKGKIPSKVIKSYYGTMDIKKINEGLLDSKI...,Strong
4,QVQLQESGPGVVKPSETLSLTCAVSGGSISDTYRWSWIRQPPGKGL...,MPAENKKVRFENTTSDKGKIPSKVIKSYYGTMDIKKINEGLLDSKI...,Strong
...,...,...,...
523,SAADEAARAASLTLLDYLVERYTDPATSEEDKRFLLRLAERLVETY...,MRIFAVFIFMTYWHLLNAFTVTVPKDLYVVEYGSNMTIECKFPVEK...,Medium
524,MEPTDEEKRGKYVAKIVAEEALKAYTETKDEEELNFLLIKLAALSE...,MTILGTTFGMVFSLLQVVSGESGYAQNGDLEDAELDDYSFSCYSQL...,Medium
525,GPLSKAEYLEEQIKFVEENKDKKELVKPKLIETLKYVGYDEEAEYY...,MTILGTTFGMVFSLLQVVSGESGYAQNGDLEDAELDDYSFSCYSQL...,Strong
526,EVNCELLNKLAEPVAKELFPENDLAKISTFYAALFFAYEGALKGKP...,MRIFAVFIFMTYWHLLNAFTVTVPKDLYVVEYGSNMTIECKFPVEK...,Weak


In [14]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
from sklearn.linear_model import LogisticRegression
import numpy as np

In [8]:
# 1. Initialize ESM-2 Model and Tokenizer
# We use the 8M parameter model (esm2_t6_8M_UR50D) for speed and efficiency.
# For higher accuracy, consider 'facebook/esm2_t33_650M_UR50D'.
model_name = "facebook/esm2_t6_8M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)

# Put model in evaluation mode
model.eval()

Loading weights: 100%|██████████| 107/107 [00:00<00:00, 3423.71it/s]


EsmForMaskedLM(
  (esm): EsmModel(
    (embeddings): EsmEmbeddings(
      (word_embeddings): Embedding(33, 320, padding_idx=1)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (rotary_embeddings): EsmRotaryEmbedding()
    (encoder): EsmEncoder(
      (layer): ModuleList(
        (0-5): 6 x EsmLayer(
          (attention): EsmAttention(
            (self): EsmSelfAttention(
              (query): Linear(in_features=320, out_features=320, bias=True)
              (key): Linear(in_features=320, out_features=320, bias=True)
              (value): Linear(in_features=320, out_features=320, bias=True)
            )
            (output): EsmSelfOutput(
              (dense): Linear(in_features=320, out_features=320, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (LayerNorm): LayerNorm((320,), eps=1e-05, elementwise_affine=True, bias=True)
          )
          (intermediate): EsmIntermediate(
            (dense): Linear(in_features=320, out_

In [15]:
def get_esm_embedding(sequence: str) -> np.ndarray:
    """Extracts a sequence-level embedding from ESM-2 by averaging token representations."""
    inputs = tokenizer(sequence, return_tensors="pt")
    
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        
    # Hidden states shape: (layers, batch_size, sequence_length, hidden_dim)
    # We take the last hidden state layer
    last_hidden_state = outputs.hidden_states[-1]
    
    # Perform mean pooling across the sequence length dimension (dim=1)
    # to get a single vector for the entire protein sequence
    sequence_embedding = torch.mean(last_hidden_state, dim=1).squeeze().numpy()
    return sequence_embedding

In [18]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

final_df = pd.read_csv("plain_training.csv")
final_df = final_df[["sequence", "target_sequence", "value"]].dropna().copy()

rng = np.random.default_rng(42)
train_indices = []
val_indices = []
test_indices = []

for _, group in final_df.groupby("target_sequence", sort=False):
    shuffled = group.sample(frac=1.0, random_state=42).index.to_numpy()
    n_samples = len(shuffled)
    raw_counts = np.array([0.7, 0.2, 0.1]) * n_samples
    split_counts = np.floor(raw_counts).astype(int)
    remainder = n_samples - split_counts.sum()
    if remainder > 0:
        fractional_parts = raw_counts - split_counts
        for split_idx in np.argsort(-fractional_parts)[:remainder]:
            split_counts[split_idx] += 1

    train_end = split_counts[0]
    val_end = train_end + split_counts[1]

    train_indices.extend(shuffled[:train_end])
    val_indices.extend(shuffled[train_end:val_end])
    test_indices.extend(shuffled[val_end:])

train_df = final_df.loc[train_indices].sample(frac=1.0, random_state=42).reset_index(drop=True)
val_df = final_df.loc[val_indices].sample(frac=1.0, random_state=42).reset_index(drop=True)
test_df = final_df.loc[test_indices].sample(frac=1.0, random_state=42).reset_index(drop=True)

print(f"Split sizes: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")
print("Label distribution in train:")
print(train_df["value"].value_counts().to_string())
print("Building paired embeddings from sequence and target_sequence...")

def build_pair_embeddings(sequences, target_sequences):
    target_cache = {}
    pair_embeddings = []

    for sequence, target_sequence in zip(sequences, target_sequences):
        sequence_embedding = get_esm_embedding(sequence)
        if target_sequence not in target_cache:
            target_cache[target_sequence] = get_esm_embedding(target_sequence)
        pair_embeddings.append(np.concatenate([sequence_embedding, target_cache[target_sequence]]))

    return np.vstack(pair_embeddings)

X_train = build_pair_embeddings(train_df["sequence"], train_df["target_sequence"])
X_val = build_pair_embeddings(val_df["sequence"], val_df["target_sequence"])
X_test = build_pair_embeddings(test_df["sequence"], test_df["target_sequence"])

print(f"Feature shape: {X_train.shape[1]} dimensions")

y_train = train_df["value"].astype(str).to_numpy()
y_val = val_df["value"].astype(str).to_numpy()
y_test = test_df["value"].astype(str).to_numpy()

classifier = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=42)
)
classifier.fit(X_train, y_train)

val_predictions = classifier.predict(X_val)
test_predictions = classifier.predict(X_test)

print(f"Validation accuracy: {accuracy_score(y_val, val_predictions):.4f}")
print(f"Test accuracy: {accuracy_score(y_test, test_predictions):.4f}")
print("\nTest classification report:")
print(classification_report(y_test, test_predictions))


Split sizes: train=369, val=107, test=52
Label distribution in train:
value
Medium    154
Strong    153
Weak       62
Building paired embeddings from sequence and target_sequence...
Feature shape: 640 dimensions
Validation accuracy: 0.5234
Test accuracy: 0.5192

Test classification report:
              precision    recall  f1-score   support

      Medium       0.50      0.46      0.48        24
      Strong       0.42      0.42      0.42        19
        Weak       0.73      0.89      0.80         9

    accuracy                           0.52        52
   macro avg       0.55      0.59      0.57        52
weighted avg       0.51      0.52      0.51        52



In [19]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_train_xgb = label_encoder.fit_transform(y_train)
y_val_xgb = label_encoder.transform(y_val)
y_test_xgb = label_encoder.transform(y_test)

xgb_model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=len(label_encoder.classes_),
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    eval_metric="mlogloss",
)

xgb_model.fit(X_train, y_train_xgb)

val_predictions_xgb = xgb_model.predict(X_val)
test_predictions_xgb = xgb_model.predict(X_test)

print(f"XGBoost validation accuracy: {accuracy_score(y_val_xgb, val_predictions_xgb):.4f}")
print(f"XGBoost test accuracy: {accuracy_score(y_test_xgb, test_predictions_xgb):.4f}")
print("\nXGBoost test classification report:")
print(classification_report(
    label_encoder.inverse_transform(y_test_xgb),
    label_encoder.inverse_transform(test_predictions_xgb)
))


XGBoost validation accuracy: 0.5514
XGBoost test accuracy: 0.5385

XGBoost test classification report:
              precision    recall  f1-score   support

      Medium       0.52      0.46      0.49        24
      Strong       0.50      0.68      0.58        19
        Weak       0.80      0.44      0.57         9

    accuracy                           0.54        52
   macro avg       0.61      0.53      0.55        52
weighted avg       0.56      0.54      0.54        52

